# CE432P: Hydraulic Engineering
## Unit 2 — Open Channel Flow & Normal Depth Solver (Day 11)
**Topic:** Uniform Flow, Manning's Equation & Automated Newton-Raphson Solver

---
### 1. Governing Equation
Manning's equation for steady uniform flow in an open channel is given by:
$$Q = \frac{1}{n} A R^{2/3} S_0^{1/2}$$

For a trapezoidal channel with bottom width $b$, side slope $m:1$ (H:V), and flow depth $y$:
- Area $A = (b + m y) y$
- Wetted Perimeter $P = b + 2 y \sqrt{1 + m^2}$
- Hydraulic Radius $R = \frac{A}{P} = \frac{(b + m y) y}{b + 2 y \sqrt{1 + m^2}}$

Because $y$ appears implicitly, we formulate the root-finding problem $f(y) = 0$:
$$f(y) = \frac{1}{n} \frac{[(b + m y) y]^{5/3}}{[b + 2 y \sqrt{1 + m^2}]^{2/3}} S_0^{1/2} - Q = 0$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def manning_residual(y, Q, n, b, m, S0):
    A = (b + m * y) * y
    P = b + 2 * y * np.sqrt(1 + m**2)
    Q_calc = (1.0 / n) * (A**(5/3) / P**(2/3)) * np.sqrt(S0)
    return Q_calc - Q

def solve_normal_depth(Q, n, b, m, S0, tol=1e-5, max_iter=100):
    # Newton-Raphson numerical solver
    y = (Q * n / (b * np.sqrt(S0)))**(3/5) # initial estimate
    for i in range(max_iter):
        f = manning_residual(y, Q, n, b, m, S0)
        if abs(f) < tol:
            return y
        # numerical derivative df/dy
        dy = 1e-6
        df = (manning_residual(y + dy, Q, n, b, m, S0) - f) / dy
        y = y - f / df
    return y

# Example Canal Design:
Q_design = 25.0   # m3/s
n_manning = 0.018 # concrete lined canal
b_width = 5.0     # bottom width in meters
side_slope = 1.5  # 1.5H : 1V
bed_slope = 0.0005 # S0 = 1 in 2000

yn = solve_normal_depth(Q_design, n_manning, b_width, side_slope, bed_slope)
A_n = (b_width + side_slope * yn) * yn
V_n = Q_design / A_n
Fr_n = V_n / np.sqrt(9.81 * (A_n / (b_width + 2 * side_slope * yn)))

print(f"Normal Depth (yn)   : {yn:.3f} m")
print(f"Flow Area (A)       : {A_n:.3f} m2")
print(f"Flow Velocity (V)   : {V_n:.3f} m/s")
print(f"Froude Number (Fr)  : {Fr_n:.3f} ({'Subcritical' if Fr_n < 1 else 'Supercritical'})")


In [ ]:
# Specific Energy Curve E vs y
y_range = np.linspace(0.3, 4.0, 300)
E_vals = []
for y in y_range:
    A = (b_width + side_slope * y) * y
    E = y + (Q_design**2) / (2 * 9.81 * A**2)
    E_vals.append(E)

E_vals = np.array(E_vals)
idx_min = np.argmin(E_vals)
yc = y_range[idx_min]
E_min = E_vals[idx_min]

plt.figure(figsize=(8, 6))
plt.plot(E_vals, y_range, lw=2.5, color="navy", label="Specific Energy Curve $E(y)$")
plt.plot([0, 6], [0, 6], "--", color="gray", alpha=0.7, label="Potential Energy Line ($E = y$)")
plt.scatter([E_min], [yc], color="red", s=80, zorder=5, label=f"Critical Depth $y_c = {yc:.2f}$ m")
plt.axhline(yn, color="green", linestyle=":", lw=2, label=f"Normal Depth $y_n = {yn:.2f}$ m")

plt.xlabel("Specific Energy $E$ (m)", fontsize=12)
plt.ylabel("Flow Depth $y$ (m)", fontsize=12)
plt.title(f"Specific Energy Curve for Q = {Q_design} m³/s", fontsize=14, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.xlim(0, 6)
plt.ylim(0, 4)
plt.legend(fontsize=11)
plt.show()
